# Notebook 14 – Performance Optimization


## 1. Vectorization

### Definition
Vectorization means performing operations on entire Pandas columns or NumPy arrays instead of processing individual values in a Python loop. These operations use optimized array routines and usually reduce Python-level overhead. It is especially useful for numerical calculations across many rows.

### Real-Life Example
A company needs to add a 10% bonus to every employee salary. Instead of calculating the bonus one employee at a time, the analyst can calculate the complete Bonus column in one operation. This makes the code shorter and easier to maintain.

### AI/ML Example
In AI/ML, vectorization is commonly used during feature engineering and numerical preprocessing. We can scale, transform, or combine thousands of feature values using column operations. This often speeds up data preparation before model training.

### Explanation
Pandas supports arithmetic directly on a whole Series. In the example, multiplying the Salary column by 0.10 creates bonus values for every row. The original salary values remain available and the resulting column can be reused.

### Python Implementation


In [1]:
import pandas as pd

df = pd.DataFrame({"Name": ["Asha", "Ravi", "Kiran"], "Salary": [30000, 40000, 35000]})

# Vectorized calculation
df["Bonus"] = df["Salary"] * 0.10
df["Total_Salary"] = df["Salary"] + df["Bonus"]
print(df)

    Name  Salary   Bonus  Total_Salary
0   Asha   30000  3000.0       33000.0
1   Ravi   40000  4000.0       44000.0
2  Kiran   35000  3500.0       38500.0


## 2. apply() vs Loops

### Definition
A Python loop processes values one by one, while Pandas apply() applies a function to a Series or along a DataFrame axis. apply() often makes custom transformations more readable, but it is not automatically faster than a loop. For simple arithmetic, vectorized operations are usually the best choice.

### Real-Life Example
A college wants to add five grace marks to student scores. The task can be performed using a loop or with apply(), depending on how the code is organized. For a simple addition, directly adding 5 to the Marks column is generally clearer.

### AI/ML Example
In AI/ML, apply() is useful for custom feature transformations that cannot easily be expressed using built-in Pandas operations. Loops can also handle complex logic but may become slow on large datasets. We should prefer vectorized methods when available and benchmark alternatives for large workloads.

### Explanation
The example compares three ways of adding five marks: a loop, apply(), and vectorization. They produce the same result for these inputs, but differ in how they execute. Do not assume apply() is always faster; actual performance depends on the operation and data size.

### Python Implementation


In [3]:
import pandas as pd

df = pd.DataFrame({"Marks": [65, 78, 89, 92]})

# Python loop
loop_result = []
for mark in df["Marks"]:
    loop_result.append(mark + 5)

# apply() with a function
apply_result = df["Marks"].apply(lambda mark: mark + 5)

# Vectorized alternative
vectorized_result = df["Marks"] + 5

print("Loop:", loop_result)
print("Apply:", apply_result.tolist())
print("Vectorized:", vectorized_result.tolist())

Loop: [70, 83, 94, 97]
Apply: [70, 83, 94, 97]
Vectorized: [70, 83, 94, 97]


## 3. Memory Optimization

### Definition
Memory optimization means reducing the amount of RAM used by a DataFrame without losing information needed for analysis. Pandas provides tools to inspect column memory usage and change data types. Choosing appropriate data types can help when working with large datasets.

### Real-Life Example
A business stores millions of customer ages as 64-bit integers even though ages are small numbers. An analyst may safely use a smaller integer type after checking the actual value range. This can reduce memory usage across the full dataset.

### AI/ML Example
In AI/ML, large training datasets can consume considerable memory during preprocessing. Reducing unnecessary memory usage can help prevent out-of-memory errors and improve data pipeline efficiency. Type conversion must preserve the required range and precision.

### Explanation
memory_usage(deep=True) estimates memory usage, including a more complete estimate for object columns. The example changes Age from int64 to int8 because all example values fit the int8 range. Always check the actual values before downcasting to avoid overflow or loss of precision.

### Python Implementation


In [5]:
import pandas as pd

df = pd.DataFrame({"Age": [21, 24, 32, 45, 29]})

print("Before:", df["Age"].dtype)
print("Memory before:", df.memory_usage(deep=True).sum(), "bytes")

# Convert to a smaller suitable integer type
df["Age"] = df["Age"].astype("int8")

print("After:", df["Age"].dtype)
print("Memory after:", df.memory_usage(deep=True).sum(), "bytes")

Before: int64
Memory before: 172 bytes
After: int8
Memory after: 137 bytes


## 4. Category Data Type

### Definition
The category data type stores repeated labels using a collection of categories and compact codes. It is especially useful when a column has relatively few unique values repeated across many rows. It may reduce memory usage and can make some categorical operations more efficient.

### Real-Life Example
A company stores a Department column with values such as HR, Sales, and IT repeated for thousands of employees. Converting the column to category avoids storing each repeated label as an independent Python string object. This can reduce memory usage in large reports.

### AI/ML Example
In AI/ML, categorical features such as city, product type, and customer segment are common. The category dtype helps represent repeated labels efficiently during exploratory analysis and preprocessing. Models may still require appropriate encoding before training.

### Explanation
astype("category") converts a suitable column to categorical storage. The example compares the memory usage of the original text column with the converted column. Category is not always smaller, especially for very small datasets or columns with mostly unique values.

### Python Implementation


In [7]:
import pandas as pd

df = pd.DataFrame({"Department": ["HR", "Sales", "IT", "HR", "Sales"] * 100})

before = df["Department"].memory_usage(deep=True)
df["Department"] = df["Department"].astype("category")
after = df["Department"].memory_usage(deep=True)

print("Data type:", df["Department"].dtype)
print("Before:", before, "bytes")
print("After:", after, "bytes")
print(df.head())

Data type: category
Before: 26232 bytes
After: 896 bytes
  Department
0         HR
1      Sales
2         IT
3         HR
4      Sales


## 5. Efficient Pandas Practices

### Definition
Efficient Pandas practices are techniques that make data processing clearer and reduce unnecessary computation or memory use. Common practices include selecting only required columns, using vectorized operations, and choosing appropriate data types. Efficiency should be balanced with correctness and readability.

### Real-Life Example
A company has a large sales report but only needs Product and Amount for a summary. Reading or selecting only those fields avoids processing unrelated information. Grouping and aggregating the required columns produces a compact report.

### AI/ML Example
In AI/ML, efficient data preparation reduces unnecessary work before training. Selecting relevant features, avoiding repeated copies, and using built-in Pandas operations can make pipelines easier to maintain. Performance improvements should be measured rather than assumed.

### Explanation
The example selects only needed columns and calculates total sales by product using groupby().sum(). This avoids a manual loop for calculating each product total. For file imports, usecols in read_csv() can prevent unnecessary columns from being loaded in the first place.

### Python Implementation


In [9]:
import pandas as pd

df = pd.DataFrame({
    "Product": ["Pen", "Book", "Pen", "Book", "Bag"],
    "Amount": [20, 100, 30, 150, 500],
    "Customer": ["Asha", "Ravi", "Kiran", "Priya", "Meena"]
})

# Select only required columns
sales = df[["Product", "Amount"]]

# Use built-in aggregation instead of a manual loop
summary = sales.groupby("Product", as_index=False)["Amount"].sum()
print(summary)

  Product  Amount
0     Bag     500
1    Book     250
2     Pen      50
